In [0]:
# import the necessary functions
# sha2 - hashes a value into a fixed-length scrambled string (256-bit)
# col - refers to an existing column in a DataFrame by name, so you can use it inside transformations
# current_timestamp - returns the exact date/time the code is run right now
# lit - inserts a fixed, constant value into every row of a new column
# to_timestamp - converts a text/string column into a real TIMESTAMP type
# concat_ws - concatenates multiple columns into a single string
# try_to_timestamp - try to parse date, if not use NULL

from pyspark.sql.functions import sha2, col, current_timestamp, lit, to_timestamp, concat_ws, try_to_timestamp

In [0]:
## category_translation 
## casa_conforto vs casa_conforto_2: (home_confort / home_comfort_2)

category_translation_silver = spark.table("olist.bronze.category_translation")
category_translation_silver = category_translation_silver.dropDuplicates(["product_category_name"])
category_translation_silver = category_translation_silver.withColumn("loaded_at", current_timestamp())
category_translation_silver = category_translation_silver.withColumn("source", lit("product_category_name_translation.csv"))
category_translation_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.category_translation")


In [0]:
## customers

customers_silver = spark.table("olist.bronze.customers")
customers_silver = customers_silver.dropDuplicates(["customer_id"]) # removes exact duplicate `customer_id` rows
customers_silver = customers_silver.withColumn("customer_sk", sha2(col("customer_id"), 256)) # generates a surrogate key (`customer_sk`) by hashing the natural ID
customers_silver = customers_silver.withColumn("loaded_at", current_timestamp())
customers_silver = customers_silver.withColumn("source", lit("olist_customers_dataset.csv"))
customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist.silver.customers")

In [0]:
## geolocation
## geolocation city "sao paulo" and "são paulo"
## deduplication drops full-row duplicates rather than using a specific key column

from pyspark.sql.functions import lower, translate

geolocation_silver = spark.table("olist.bronze.geolocation")
geolocation_silver = geolocation_silver.dropDuplicates()
geolocation_silver = geolocation_silver.withColumn(
    "geolocation_city_clean",
    lower(translate(col("geolocation_city"), "áàâãéèêíïóôõöúçñ", "aaaaeeeiiooooucn"))
)
geolocation_silver = geolocation_silver.withColumn("loaded_at", current_timestamp())
geolocation_silver = geolocation_silver.withColumn("source", lit("olist_geolocation_dataset.csv"))
geolocation_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.geolocation")


In [0]:
## order_items 
## no single natural ID - each row is identified by (order_id + order_item_id) together, so the surrogate key is a hash of both combined
## `shipping_limit_date` is converted to a correct TIMESTAMP

order_items_silver = spark.table("olist.bronze.order_items")
order_items_silver = order_items_silver.dropDuplicates(["order_id", "order_item_id"])
order_items_silver = order_items_silver.withColumn(
    "order_item_sk", sha2(concat_ws("||", col("order_id"), col("order_item_id")), 256)
)
order_items_silver = order_items_silver.withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
order_items_silver = order_items_silver.withColumn("loaded_at", current_timestamp())
order_items_silver = order_items_silver.withColumn("source", lit("olist_order_items_dataset.csv"))
order_items_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.order_items")


In [0]:
## orders
## 'canceled' orders still have a order_delivered_customer_date
## deduplicates on `order_id` and converts all five date columns from raw text into proper TIMESTAMP types

orders_silver = spark.table("olist.bronze.orders")
orders_silver = orders_silver.dropDuplicates(["order_id"])
orders_silver = orders_silver.withColumn("order_sk", sha2(col("order_id"), 256))
orders_silver = orders_silver.withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
orders_silver = orders_silver.withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
orders_silver = orders_silver.withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
orders_silver = orders_silver.withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
orders_silver = orders_silver.withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
orders_silver = orders_silver.withColumn("loaded_at", current_timestamp())
orders_silver = orders_silver.withColumn("source", lit("olist_customers_dataset.csv"))
orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist.silver.orders")


In [0]:
## payments 
# a single order can have multiple payments, so the key is (order_id + payment_sequential) together

payments_silver = spark.table("olist.bronze.payments")
payments_silver = payments_silver.dropDuplicates(["order_id", "payment_sequential"])
payments_silver = payments_silver.withColumn(
    "payment_sk", sha2(concat_ws("||", col("order_id"), col("payment_sequential")), 256)
)
payments_silver = payments_silver.withColumn("loaded_at", current_timestamp())
payments_silver = payments_silver.withColumn("source", lit("olist_order_payments_dataset.csv"))
payments_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.payments")


In [0]:
## products
## removes duplicate product ids and generates a surrogate key
products_silver = spark.table("olist.bronze.products")
products_silver = products_silver.dropDuplicates(["product_id"])
products_silver = products_silver.withColumn("product_sk", sha2(col("product_id"), 256))
products_silver = products_silver.withColumn("loaded_at", current_timestamp())
products_silver = products_silver.withColumn("source", lit("olist_products_dataset.csv"))
products_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.products")

In [0]:
## reviews
## review_id: line 14 - incorrect, 2018-02-16 00:00:00,2018-02-20 10:52:22
## lots of inconsistencies in review_id
## removes duplicates on `review_id` and converts date columns to TIMESTAMP


reviews_silver = spark.table("olist.bronze.reviews")
reviews_silver = reviews_silver.dropDuplicates(["review_id"])
reviews_silver = reviews_silver.withColumn("review_sk", sha2(col("review_id"), 256))
reviews_silver = reviews_silver.withColumn("review_creation_date", try_to_timestamp(col("review_creation_date")))
reviews_silver = reviews_silver.withColumn("review_answer_timestamp", try_to_timestamp(col("review_answer_timestamp")))
reviews_silver = reviews_silver.withColumn("loaded_at", current_timestamp())
reviews_silver = reviews_silver.withColumn("source", lit("olist_order_reviews_dataset.csv"))
reviews_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.reviews")


In [0]:
## sellers
## seller_city: 1 numerical value, 04482255
## removes duplicates of seller_id and generates a surrogate key

sellers_silver = spark.table("olist.bronze.sellers")
sellers_silver = sellers_silver.dropDuplicates(["seller_id"])
sellers_silver = sellers_silver.withColumn("seller_sk", sha2(col("seller_id"), 256))
sellers_silver = sellers_silver.withColumn("loaded_at", current_timestamp())
sellers_silver = sellers_silver.withColumn("source", lit("olist_sellers_dataset.csv"))
sellers_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.sellers")


In [0]:
%sql
-- constraints

-- tables with a surrogate key
ALTER TABLE olist.silver.customers ALTER COLUMN customer_sk SET NOT NULL;
ALTER TABLE olist.silver.orders ALTER COLUMN order_sk SET NOT NULL;
ALTER TABLE olist.silver.products ALTER COLUMN product_sk SET NOT NULL;
ALTER TABLE olist.silver.sellers ALTER COLUMN seller_sk SET NOT NULL;
ALTER TABLE olist.silver.order_items ALTER COLUMN order_item_sk SET NOT NULL;
ALTER TABLE olist.silver.payments ALTER COLUMN payment_sk SET NOT NULL;

-- NULL review_id
DELETE FROM olist.silver.reviews
WHERE review_sk IS NULL;
ALTER TABLE olist.silver.reviews ALTER COLUMN review_sk SET NOT NULL;

-- tables with no surrogate key 
ALTER TABLE olist.silver.geolocation ALTER COLUMN geolocation_zip_code_prefix SET NOT NULL;
ALTER TABLE olist.silver.category_translation ALTER COLUMN product_category_name SET NOT NULL;